# Qwen3.8-27B quantized Colab pilot

This is an unchanged-model hardware and inference check on a free Colab GPU. It does not fine-tune the model or count as Research 3 self-preservation evidence.

The [official Unsloth 27B conversational fine-tuning notebook](https://github.com/unslothai/notebooks/blob/main/nb/Qwen3.8_%2827B%29-Conversational.ipynb) is the prepared training route for larger hardware. It requires two T4 GPUs; this free Colab runtime offered one. All model-visible test text here is fictional and no model output is executed as code.


In [ ]:
import torch, subprocess, shutil, platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name)
    print("VRAM GiB:", round(p.total_memory / 2**30, 2))
print("Free disk GiB:", round(shutil.disk_usage("/content").free / 2**30, 2))
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"], capture_output=True, text=True).stdout)


In [ ]:
from pathlib import Path
import subprocess, tarfile
release = "b11266"
archive = Path("/content/llama-cuda.tar.gz")
root = Path("/content/llama-cuda")
url = f"https://github.com/ggml-org/llama.cpp/releases/download/{release}/llama-{release}-bin-ubuntu-cuda-12.8-x64.tar.gz"
if not archive.exists():
    subprocess.run(["curl", "-fL", "--retry", "3", "-o", str(archive), url], check=True)
root.mkdir(exist_ok=True)
with tarfile.open(archive) as bundle:
    bundle.extractall(root, filter="data")
binaries = list(root.rglob("llama-cli"))
assert len(binaries) == 1, binaries
llama_cli = binaries[0]
print("llama.cpp:", llama_cli)
subprocess.run([str(llama_cli), "--version"], check=True)


In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
repo = "unsloth/Qwen3.8-27B-GGUF"
revision = "4ca720788d1e01f1bff70c033e0d0028fd02e502"
filename = "Qwen3.8-27B-UD-IQ3_S.gguf"
model_path = Path(hf_hub_download(repo_id=repo, filename=filename, revision=revision, local_dir="/content/model"))
print("Model:", model_path)
print("Size GiB:", round(model_path.stat().st_size / 2**30, 2))


In [ ]:
import time, subprocess
prompt = "This is a fictional inventory task. There are 13 blue pens and 3 more arrive. Give only the total number of blue pens."
cmd = [str(llama_cli), "-m", str(model_path), "-ngl", "99", "-c", "512", "-n", "24", "--temp", "0", "--single-turn", "--simple-io", "--reasoning-budget", "0", "--no-display-prompt", "-p", prompt]
started = time.perf_counter()
try:
    run = subprocess.run(cmd, input="", capture_output=True, text=True, timeout=180)
    print("Exit code:", run.returncode)
    print("Elapsed seconds:", round(time.perf_counter() - started, 1))
    print("Response tail:\n", run.stdout[-2500:])
    print("Runtime details tail:\n", run.stderr[-2500:])
except subprocess.TimeoutExpired as error:
    print("Timed out after", round(time.perf_counter() - started, 1), "seconds")
    print("Partial output:\n", (error.stdout or b"")[-1500:])
    print("Partial runtime details:\n", (error.stderr or b"")[-1500:])
